# Domino-RL — evaluation notebook

Plots and tables for the report, built only from the artefacts on disk:

* `results/training_*.csv` — the convergence mirror written during training
  (one row per finished deal: `step,episode,total_reward,window_avg`)
* `results/eval_*.csv` — the head-to-head matrix written by `test.py`
* `results/rule_weight_sweep.csv` — the rule-bot weight sweep (ablation A7)
* `runs/benchmark.json` — the measured training throughput

Regenerate everything with:

```bash
uv run python train.py --steps 4096 --n-envs 2 --run-name smoke   # a training run
uv run python test.py --matchups all --deals 300                  # the evaluation matrix
uv run python tools/sweep_rule_weights.py                         # the weight sweep
uv run python tools/bench_throughput.py                           # the throughput numbers
```

Every cell below degrades gracefully when its input file is missing, so the notebook can be
executed before training has produced anything.

In [ ]:
from __future__ import annotations

import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["figure.dpi"] = 110
RESULTS = pathlib.Path("results")
RUNS = pathlib.Path("runs")
WINDOW = 100


def load_glob(pattern: str, results: pathlib.Path = RESULTS) -> pd.DataFrame:
    """Concatenate every CSV matching ``pattern`` (empty frame when there are none)."""
    frames = []
    for path in sorted(results.glob(pattern)):
        frame = pd.read_csv(path)
        frame["source"] = path.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


training = load_glob("training_*.csv")
evaluation = load_glob("eval_*.csv")
print("training rows:", len(training), "| evaluation rows:", len(evaluation))

## 1. Training convergence — total (undiscounted) reward per episode

The assignment asks for convergence to be read from the **total reward per episode**, with a
sliding-window average when the series is noisy. Both are plotted below; the window average is
the 100-episode mean of the same undiscounted totals, never a discounted return.

In [ ]:
if training.empty:
    print("no results/training_*.csv yet — run train.py first")
else:
    figure, axis = plt.subplots(figsize=(10, 4.5))
    for run, frame in training.groupby("source"):
        frame = frame.sort_values("episode")
        axis.plot(frame["episode"], frame["total_reward"], alpha=0.25, linewidth=0.8,
                  label=f"{run} per episode")
        axis.plot(frame["episode"], frame["window_avg"], linewidth=2.0,
                  label=f"{run} {WINDOW}-episode average")
    axis.axhline(0.0, color="black", linewidth=0.8, linestyle=":")
    axis.set_xlabel("episode (deal)")
    axis.set_ylabel("total undiscounted reward (points)")
    axis.set_title("Training convergence")
    axis.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    summary = (training.groupby("source")["total_reward"]
               .agg(["count", "mean", "std", "min", "max"]).round(3))
    summary.columns = ["episodes", "mean reward", "std", "min", "max"]
    display(summary)

## 2. Head-to-head evaluation matrix

`test.py` writes one row per matchup. Win rate comes with a **Wilson 95 % interval** so a
modest sample is not over-read; points per deal capture the margin; the match series captures
the game as it is actually played to 100 points.

In [ ]:
if evaluation.empty:
    print("no results/eval_*.csv yet — run test.py first")
else:
    columns = ["matchup", "deals", "wins_a", "wins_b", "win_rate_a", "ci_low", "ci_high",
               "points_per_deal_a", "points_per_deal_b", "match_win_rate_a", "matches_played",
               "block_rate", "void_rate", "mean_pips_left_a", "ms_per_action", "seed", "source"]
    table = evaluation[[c for c in columns if c in evaluation.columns]].copy()
    for column in table.columns:
        if table[column].dtype.kind == "f":
            table[column] = table[column].round(3)
    display(table)

In [ ]:
if not evaluation.empty:
    frame = evaluation.dropna(subset=["win_rate_a"]).sort_values("win_rate_a")
    errors = [frame["win_rate_a"] - frame["ci_low"], frame["ci_high"] - frame["win_rate_a"]]
    figure, axis = plt.subplots(figsize=(9, 4))
    axis.barh(frame["matchup"], frame["win_rate_a"], xerr=errors, capsize=4, color="#4a86c8")
    axis.axvline(0.5, color="black", linestyle="--", linewidth=0.9, label="coin flip")
    axis.set_xlim(0, 1)
    axis.set_xlabel("deals won by agent A (Wilson 95 % CI)")
    axis.set_title("Win rates per matchup")
    axis.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
if not evaluation.empty:
    frame = evaluation.copy()
    positions = np.arange(len(frame))
    width = 0.38
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    axes[0].bar(positions - width / 2, frame["points_per_deal_a"], width,
                label="agent A", color="#4a86c8")
    axes[0].bar(positions + width / 2, frame["points_per_deal_b"], width,
                label="agent B", color="#c86f4a")
    axes[0].set_xticks(positions)
    axes[0].set_xticklabels(frame["matchup"], rotation=20, ha="right", fontsize=8)
    axes[0].set_ylabel("points banked per deal")
    axes[0].set_title("Margin per deal")
    axes[0].legend(fontsize=8)

    axes[1].bar(positions - width / 2, frame["block_rate"], width,
                label="blocked deals", color="#6aa84f")
    axes[1].bar(positions + width / 2, frame["void_rate"], width,
                label="void deals", color="#b45f06")
    axes[1].set_xticks(positions)
    axes[1].set_xticklabels(frame["matchup"], rotation=20, ha="right", fontsize=8)
    axes[1].set_ylabel("fraction of rounds")
    axes[1].set_title("Blocked and void rates")
    axes[1].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

## 3. Rule-bot weight sweep (ablation A7)

The rule-based baseline's seven weights were chosen by a coordinate-ascent sweep on seeds that
are disjoint from the test gate's seeds, so the baseline's strength is a measurement rather
than a hand-picked constant.

In [ ]:
sweep_path = RESULTS / "rule_weight_sweep.csv"
if not sweep_path.exists():
    print("no results/rule_weight_sweep.csv — run tools/sweep_rule_weights.py")
else:
    sweep = pd.read_csv(sweep_path)
    display(sweep.head(5))
    if {"mean_win_rate", "weights"} <= set(sweep.columns):
        figure, axis = plt.subplots(figsize=(9, 4))
        axis.barh([f"rank {i + 1}" for i in range(min(10, len(sweep)))],
                  sweep["mean_win_rate"].head(10), color="#6aa84f")
        axis.set_xlabel("mean win rate vs random (sweep seeds)")
        axis.set_title("Rule-bot weight sweep — top candidates")
        plt.tight_layout()
        plt.show()

## 4. Measured training throughput

The training budget is set from these numbers, not from an estimate: `tools/bench_throughput.py`
measures the raw engine and the end-to-end learner with the real vectorised environment.

In [ ]:
bench_path = RUNS / "benchmark.json"
if not bench_path.exists():
    print("no runs/benchmark.json — run tools/bench_throughput.py")
else:
    payload = json.loads(bench_path.read_text(encoding="utf-8"))
    engine = payload["engine"]
    print(f"raw engine: {engine['deals_per_second']:.0f} deals/s, "
          f"{engine['moves_per_second']:.0f} moves/s ({engine['moves_per_deal']:.1f} moves/deal)")
    rows = []
    for row in payload["learners"]:
        rows.append({"opponent": row["opponent"], "n_envs": row["n_envs"],
                     "steps": row["steps"], "seconds": round(row["seconds"], 1),
                     "steps/s": round(row["steps_per_second"])})
        for total in (10_000_000, 30_000_000, 100_000_000):
            rows.append({"opponent": f"  -> {total // 1_000_000} M steps at this rate",
                         "n_envs": "", "steps": "", "seconds": "",
                         "steps/s": f"{total / row['steps_per_second'] / 3600:.1f} h"})
    display(pd.DataFrame(rows))

## 5. Reading the results

What the numbers mean, and what they do not:

* **Win rate** is per *deal*; the Wilson interval is what you should quote, not the point
  estimate, whenever `deals` is in the low hundreds.
* **Points per deal** is the quantity the agent's reward optimises, so it is the fairest
  single-number comparison against the rule bot.
* **Match win rate** comes from far fewer samples than the deal statistics — read it as
  indicative only.
* **Blocked and void rates** show whether the agent learned the endgame at all: the blocked
  deals are decided by the fewest-pips rule, which is a different skill from shedding tiles.
* **The convergence curve** is the total undiscounted reward per episode; a rising
  100-episode average with a flat per-episode spread is what a healthy run looks like. A
  window average that plateaus is the signal to stop, not to keep spending hours.
* **Caveat on the shipped artefacts**: the checkpoints in `models/` come from short smoke runs,
  so any `rl_*` row produced from them reflects an almost-untrained agent. Re-run
  `train.py` for the budget in the design spec's section 14 and then re-run `test.py`.
